# Kamath × Menon × Siletti × ASAP scVI+scANVI Integration Explorer

Interactive exploration of outputs from `04_integrate_kamath_menon_siletti_asap.py`.

- **Section 1** — UMAP explorer: colour dots by any obs column. Use lasso-tool to select barcodes from DA neurons.
- **Section 2** — Label agreement heatmap: scANVI predictions vs native cell type labels
- **Section 3** - Read Depth QC — UMI counts & genes measured per cell type

## Imports

In [ ]:
import os
import re
import sys
import glob
import functools
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display
from pathlib import Path
import warnings

pio.renderers.default = 'plotly_mimetype'
warnings.filterwarnings("ignore", message=".*deprecated.*")

## Inputs

In [ ]:
# Change downsampling_strategy and kamath_representation
# The rest should be automatically loaded
# ------------
# Expected folder name and string in h5ad with results from integrate_*.py. Use:
# downsampling_strategy = "asap96k_kamath99k_menon99k_siletti100k_hvg_u_da_markers" (All 4-sources: ASAP, Kamath, Menon, Siletti)
# downsampling_strategy = "asap96k_menon99k_siletti100k_list_asap96k_menon99k_siletti100k_hvg_minus_da_markers_gene_list_no_kamath_no_kamath" (Exclude Kamath)
downsampling_strategy = "asap96k_kamath97k_menon99k_siletti100k_hvg_u_da_markers" # Downsampling and HVG strategy
#
# Kamath representation:
# Kamath_all_DAs: Original strategy, including all DA's from Kamath and topped up with remaining cells
# Kamath_random: All cell types from Kamath randomly downsampled (i.e. same proportions as originally provided by authors)
# Kamath_min200: Each DA subtype with at lease 200 cells to have enough for downstream deep learning, then randomly downsample
kamath_representation = "Kamath_all_DAs" # (either Kamath_random, Kamath_all_DAs Kamath_min200)
# ------------

repo_name = "spatial-sc-rnaseq-integration-wf"  # needed for get_repo_root to work in both unpromoted and live repo notebooks
sys.path.insert(0, str(Path.cwd().parent / "utils"))
from notebook_utils import get_repo_root # must come after the utils sys.path.insert above
repo_root = get_repo_root(repo_name)

h5ad_name = 'integrated_' + downsampling_strategy + '.h5ad'
OUTPUT_DIR = str(repo_root / "reference_building" / "da_neurons_identification_analysis" / "output" / kamath_representation / downsampling_strategy)
REFERENCE_DATA_DIR = str(repo_root / "reference_building" / "da_neurons_identification_analysis" / "reference_data")  # Intermediate H5ADs used as inputs for 04_integrate_kamath_menon_siletti_asap.py

## NOTE: Siletti datased labels provided by the Allen Institute (i.e including DA VGLUT2, not the ones in the CELLxGENE H5AD)
REFERENCE_DATA_DIR_PATH = repo_root / "reference_building" / "da_neurons_identification_analysis" / "reference_data"
REFERENCE_DATA_DIR = str(REFERENCE_DATA_DIR_PATH)
ALLEN_CELL_METADATA_PATH = str(REFERENCE_DATA_DIR_PATH / "allen-brain-cell-atlas" / "metadata" / "WHB-10Xv3" / "20241115" / "cell_metadata.csv.gz")
ALLEN_MEMBERSHIP_PATH = str(REFERENCE_DATA_DIR_PATH / "allen-brain-cell-atlas" / "metadata" / "WHB-taxonomy" / "20240330" / "cluster_to_cluster_annotation_membership.csv.gz")

## Utils

In [ ]:
# ------------- REPO WIDE utils/ -------------
from notebook_utils import (
    dropdown,
    inject_big_dropdown_css,
    plot_umaps_by_gene,
    plot_umaps_categ,
    colorize_label_html,
    crosstab_heatmap_figure,
    crosstab_sort_order,
)

# ------------- KAMATH ANNOTATIONS -------------
# DA subtype vulnerability labels (Kamath et al., Nat Neurosci 2022, Figure 3A & 3B)
# Vulnerability statistical values were extracted from the main text.
_DA_PD_SUFFIXES = {
    'SOX6_AGTR1':        'SOX6_AGTR1_vulnerable',  # significantly vulnerable (OR = −0.82, FDR-adjusted P < 0.05)
    'SOX6_DDT':          'SOX6_DDT',
    'SOX6_PART1':        'SOX6_PART1_down_ns',     # on the vulnerable side, but not cited as significant (no full stats tables were found)
    'SOX6_GFRA2':        'SOX6_GFRA2_up_ns',       # on the resilent side, but not cited as significant (no full stats tables were found)
    'CALB1_CALCR':       'CALB1_CALCR',
    'CALB1_RBP4':        'CALB1_RBP4',
    'CALB1_PPP1R17':     'CALB1_PPP1R17',
    'CALB1_CRYM_CCDC68': 'CALB1_CRYM_CCDC68',
    'CALB1_GEM':         'CALB1_GEM_resistant',    # significantly resistant (OR = 1.46, FDR-adjusted P < 0.05)
    'CALB1_TRHR':        'CALB1_TRHR_resistant',   # significantly resistant (OR = 0.84, FDR-adjusted P < 0.05)
}


# ------------- SILETTI/ALLEN ANNOTATIONS -------------
# subcluster -> cluster is a verified stable many-to-one mapping in Allen's taxonomy.
def _load_siletti_subcluster_to_cluster():
    membership = pd.read_csv(ALLEN_MEMBERSHIP_PATH, keep_default_na=False)
    sub = membership.loc[
        membership['cluster_annotation_term_set_name'] == 'subcluster',
        ['cluster_alias', 'cluster_annotation_term_name'],
    ].rename(columns={'cluster_annotation_term_name': 'subcluster_name'})
    clu = membership.loc[
        membership['cluster_annotation_term_set_name'] == 'cluster',
        ['cluster_alias', 'cluster_annotation_term_name'],
    ].rename(columns={'cluster_annotation_term_name': 'cluster_name'})
    merged = sub.merge(clu, on='cluster_alias').drop_duplicates('subcluster_name')
    return merged.set_index('subcluster_name')['cluster_name']


# ------------- ALL DATASET ANNOTATIONS -------------
_NATIVE_COLS_BY_SOURCE = {
    'kamath': ('cell_type_scanvi',       # Kamath cell type used by 04_integrate_kamath_menon_siletti_asap.py to train scANVI for label propagation 
               'da_subtype'),            # DA-only subset of cell_type_scanvi
    'menon': ('menon_cell_type',         # Menon full cell type description
              'menon_author_cell_type'), # Abbreviated description. Maps 1:1 with menon_cell_type
    'siletti': (
        'siletti_cxg_cell_type', 'siletti_cxg_author_cell_type', # Siletti, originally deposited in CELLxGENE
        'siletti_allen_da_vglut2_label', 'siletti_allen_cluster', 'siletti_allen_cell_type', # Re-annotated by the Allen Institute (incl. "DA VGLUT2")
    ),
    'asap_cohort': ('pmdbs_sc_rnaseq.C_scANVI',   # scANVI-propagated labels from the ASAP pmdbs_sc_rnaseq pipeline (not from 04_integrate_kamath_menon_siletti_asap.py)
                    'pmdbs_sc_rnaseq.cell_type'),  # MMC / Allen Brain Atlas labels from the ASAP pmdbs_sc_rnaseq pipeline (not from 04_integrate_kamath_menon_siletti_asap.py)
}


# ------------- DROPDOWNS -------------
inject_big_dropdown_css()
source_dropdown = dropdown(
    big=True,
    options=['all sources', 'kamath', 'menon', 'siletti', 'asap_cohort'],
    description='Source:',
)
qc_source_dropdown = dropdown(
    big=True,
    options=['kamath', 'menon', 'siletti', 'asap_cohort'],
    description='Source:',
    layout=widgets.Layout(width='300px', height='56px'),
)
qc_sort_by_dropdown = dropdown(
    options=['median', 'alphanumeric'],
    description='Sort by:',
    layout=widgets.Layout(width='300px'),
)
_plot_umaps_categ = functools.partial(
    plot_umaps_categ,
    native_cols_by_source=_NATIVE_COLS_BY_SOURCE,
    display_label_remap=_DA_PD_SUFFIXES,
)


# ------------- AGREEMENT HEATMAPS -------------
# DA-related prefixes highlighted red in plot labels.
_DA_LABEL_PREFIXES = ('CALB1_', 'SOX6_', 'DA VGLUT2', 'Splat_395', 'Dopaminergic')
_colorize_label = functools.partial(colorize_label_html, prefixes=_DA_LABEL_PREFIXES)


def _plot_crosstab_heatmap(
    obs_df, row_col, col_col, normalize, show_values=False,
    color_scale='Blues', remap_da=True, sort_by='counts (max to min)',
    marginal_frac=0.10,
):
    """Cross-tabulate two obs columns as an interactive heatmap with log10 marginal histograms."""
    remap = _DA_PD_SUFFIXES if remap_da else {}
    pair = pd.DataFrame({
        'rows': obs_df[row_col].map(lambda x: remap.get(x, x)),
        'cols': obs_df[col_col],
    }).dropna()

    if pair.empty:
        print(f'No non-null pairs for ({row_col}, {col_col}).')
        return

    raw = pd.crosstab(pair['rows'], pair['cols']).astype(float)
    # Row-normalize before clustering -- raw crosstab counts span orders of
    # magnitude across cell types and would otherwise dominate the clustering.
    row_order, col_order = crosstab_sort_order(
        raw, raw.sum(axis=1), raw.sum(axis=0), sort_by, normalize_rows=True,
    )

    ct = raw.copy()
    if normalize == 'rows (%)':
        ct = ct.div(ct.sum(axis=1), axis=0) * 100
        colorbar_title = 'rows (%)'
        text_fmt = '.1f'
    elif normalize == 'columns (%)':
        ct = ct.div(ct.sum(axis=0), axis=1) * 100
        colorbar_title = 'columns (%)'
        text_fmt = '.1f'
    else:
        colorbar_title = 'cells (n)'
        text_fmt = '.0f'

    ct = ct.loc[row_order, col_order]
    row_totals = raw.sum(axis=1).loc[row_order]
    col_totals = raw.sum(axis=0).loc[col_order]

    marginal_px = round(140 * marginal_frac / 0.18)  # scale fixed padding with marginal_frac
    height = max(500, len(ct) * 18 + 220) + marginal_px
    width  = max(650, len(ct.columns) * 20 + 280) + marginal_px

    text = ct.map(lambda v: format(v, text_fmt)).values if show_values else None
    margin_l, margin_b, margin_t, margin_r = 220, 150, 60, 60

    fig = crosstab_heatmap_figure(
        ct, row_order, col_order, row_totals, col_totals, height, width,
        heatmap_kwargs=dict(
            colorscale=color_scale,
            text=text, texttemplate='%{text}' if show_values else None,
            colorbar=dict(title=colorbar_title),
            hovertemplate=f'{row_col}=%{{y}}<br>{col_col}=%{{x}}<br>{colorbar_title}=%{{z}}<extra></extra>',
        ),
        row_ticktext=[_colorize_label(l) for l in row_order],
        col_ticktext=[_colorize_label(l) for l in col_order],
        marginal_frac=marginal_frac,
        title=f'{row_col}  ×  {col_col}  ({colorbar_title})  —  marginals: log10(cell count)',
        margin=dict(l=margin_l, b=margin_b, t=margin_t, r=margin_r),
    )

    # Axis titles placed a fixed pixel distance from the canvas edge (not a
    # fixed paper-fraction, which drifts off-canvas as the plot grows).
    plot_h = height - margin_t - margin_b
    plot_w = width - margin_l - margin_r
    x_title_y = -(margin_b - 20) / plot_h
    y_title_x = -(margin_l - 20) / plot_w

    heatmap_x_center = (marginal_frac + 1) / 2
    heatmap_y_center = (marginal_frac + 1) / 2
    fig.add_annotation(
        text=f'<b>{col_col}</b>', xref='paper', yref='paper',
        x=heatmap_x_center, y=x_title_y, xanchor='center', yanchor='top',
        showarrow=False, font=dict(size=13),
    )
    fig.add_annotation(
        text=f'<b>{row_col}</b>', xref='paper', yref='paper',
        x=y_title_x, y=heatmap_y_center, xanchor='center', yanchor='middle',
        textangle=-90, showarrow=False, font=dict(size=13),
    )

    display(fig)

_COLOR_SCALE_BY_SOURCE = {
    'kamath': 'Purples',
    'menon': 'Blues',
    'siletti': 'Greens',
    'asap_cohort': 'Oranges',
}


def label_agreement_widget(adata, source, pred_cols=('cell_type_pred', 'cell_type_scanvi')):
    """Interactive predicted-vs-native label cross-tab heatmap, for one source."""
    obs_source = 'asap' if source == 'asap_cohort' else source  # adata.obs['source'] uses 'asap', not 'asap_cohort'
    obs = adata.obs[adata.obs['source'] == obs_source]
    row_opts = [c for c in pred_cols if c in obs.columns]
    col_opts = [c for c in _NATIVE_COLS_BY_SOURCE.get(source, ()) if c in obs.columns]

    print(f'{source} cells: {len(obs):,}')
    if not row_opts:
        print(f'No prediction columns found for source={source!r}.')
        return
    if not col_opts:
        print(f'No native label columns found for source={source!r}.')
        return

    @widgets.interact(
        row_col=widgets.Dropdown(
            options=row_opts,
            description='Rows (y-axis):',
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='420px'),
        ),
        col_col=widgets.Dropdown(
            options=col_opts,
            description='Columns (x-axis):',
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='420px'),
        ),
        normalize=widgets.Dropdown(
            options=['rows (%)', 'columns (%)', 'raw counts'],
            description='Colour encodes:',
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='420px'),
        ),
        sort_by=widgets.Dropdown(
            options=['counts (max to min)', 'alphanumeric', 'cluster'],
            description='Sort by:',
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='420px'),
        ),
    )
    def _plot(row_col, col_col, normalize, sort_by):
        _plot_crosstab_heatmap(
            obs, row_col, col_col, normalize,
            color_scale=_COLOR_SCALE_BY_SOURCE.get(source),
            sort_by=sort_by,
        )


# ------------- QC BOXPLOT -------------
# Multiplies the base plot width (1500px).
BOXPLOT_WIDTH_RATIO = 0.5
ASAP_CUTOFF_N_GENES = 10000
ASAP_CUTOFF_UMI = 100000


_INTERMEDIATE_GLOBS = {
    'kamath': f'{REFERENCE_DATA_DIR}/kamath/intermediate/kamath_merged_da_all_non_da_*.h5ad',
    'menon': f'{REFERENCE_DATA_DIR}/menon/intermediate/*_sn_only*.h5ad',
    'siletti': f'{REFERENCE_DATA_DIR}/siletti/intermediate/midbrain_da_neurons*_cells*.h5ad',
    'asap_cohort': f'{REFERENCE_DATA_DIR}/asap_cohort/pmdbs_sc_rnaseq/cohort_analysis/*/*/intermediate/*_rand_*.h5ad',
}
_NATIVE_GROUP_COL_BY_SOURCE = {
    'kamath': 'cell_type_scanvi',
    'menon': 'menon_author_cell_type',
    'siletti': 'siletti_cxg_cell_type',
    'asap_cohort': 'pmdbs_sc_rnaseq.cell_type',
}


_SOURCE_KEY_BY_NAME = {'kamath': 'kamath', 'menon': 'menon', 'siletti': 'siletti', 'asap_cohort': 'asap'}


def _parse_downsampling_targets(downsampling):
    """Parse {'asap': 50000, 'kamath': 52000, ...} from 'asap50k_kamath52k_...'."""
    return {m.group(1): int(m.group(2)) * 1000 for m in re.finditer(r'([a-z]+)(\d+)k', downsampling)}


def _matching_intermediate_cache(source, downsampling):
    """Intermediate cache for `source` whose cell count best matches `downsampling`."""
    paths = glob.glob(_INTERMEDIATE_GLOBS[source])
    if not paths:
        return None
    if len(paths) == 1:
        return paths[0]
    target = _parse_downsampling_targets(downsampling).get(_SOURCE_KEY_BY_NAME[source])
    if target is None:
        return max(paths, key=lambda p: sc.read_h5ad(p, backed='r').n_obs)
    return min(paths, key=lambda p: abs(sc.read_h5ad(p, backed='r').n_obs - target))


def _boxplot_pair_by_celltype(per_cell_df, group_col, source, sort_by='median'):
    """UMI counts / genes measured box plots per group, DA labels in red. sort_by: 'median' or 'alphanumeric'."""
    if sort_by == 'alphanumeric':
        order = sorted(per_cell_df[group_col].unique())
    else:
        order = (
            per_cell_df.groupby(group_col, observed=True)['total_counts']
            .median().sort_values(ascending=False).index.tolist()
        )
    order = order[::-1]  # plotly renders categoryarray bottom-to-top

    fig = make_subplots(
        rows=1, cols=2, shared_yaxes=True, horizontal_spacing=0.03,
        subplot_titles=('UMI counts', 'genes measured'),
    )
    fig.add_trace(go.Box(
        x=per_cell_df['total_counts'], y=per_cell_df[group_col], orientation='h',
        boxpoints=False, line_width=1, marker_color='#3D1F61', fillcolor='#3D1F61',
    ), row=1, col=1)
    fig.add_trace(go.Box(
        x=per_cell_df['n_genes'], y=per_cell_df[group_col], orientation='h',
        boxpoints=False, line_width=1, marker_color='#D98E04', fillcolor='#D98E04',
    ), row=1, col=2)

    fig.update_yaxes(
        categoryorder='array', categoryarray=order,
        tickmode='array', tickvals=order, ticktext=[_colorize_label(l) for l in order],
        row=1, col=1,
    )
    fig.update_xaxes(type='log', row=1, col=1)
    fig.update_xaxes(type='log', row=1, col=2)
    height = max(400, len(order) * 18 + 150)
    width = round(1500 * BOXPLOT_WIDTH_RATIO)
    fig.update_layout(
        height=height, width=width, showlegend=False,
        title=f'{source}: read depth per cell type (native transcriptome)',
        margin=dict(l=260, t=80, b=40, r=40),
    )
    fig.add_vline(
        x=ASAP_CUTOFF_UMI, row=1, col=1,
        line_dash='dot', line_color='red', line_width=1.5,
        annotation_text=f'{ASAP_CUTOFF_UMI:,}', annotation_position='top',
        annotation_font_color='red',
    )
    fig.add_vline(
        x=ASAP_CUTOFF_N_GENES, row=1, col=2,
        line_dash='dot', line_color='red', line_width=1.5,
        annotation_text=f'{ASAP_CUTOFF_N_GENES:,}', annotation_position='top',
        annotation_font_color='red',
    )
    display(fig)


def qc_boxplot_widget(source, sort_by='median'):
    """UMI-count and genes-measured box plots per native cell type, for `source`."""
    path = _matching_intermediate_cache(source, downsampling_strategy)
    if path is None:
        print(f'No intermediate cache found for source={source!r}.')
        return
    group_col = _NATIVE_GROUP_COL_BY_SOURCE[source]
    print(f'Loading {path} ...')
    a = sc.read_h5ad(path)
    if group_col not in a.obs.columns:
        print(f'Column {group_col!r} not found in {path}.')
        return

    counts = a.layers['counts']
    df = pd.DataFrame({
        'total_counts': np.asarray(counts.sum(axis=1)).ravel(),
        'n_genes': np.asarray((counts > 0).sum(axis=1)).ravel(),
        group_col: a.obs[group_col].astype(str).values,
    })

    if source == 'siletti' and 'siletti_allen_da_vglut2' in a.obs.columns:
        da_mask = a.obs['siletti_allen_da_vglut2'].astype(bool).values
        da_rows = df[da_mask].copy()
        da_rows[group_col] = 'DA VGLUT2 (Allen)'
        df = pd.concat([df, da_rows], ignore_index=True)

    print(f'{source}: {a.n_obs:,} cells, {df[group_col].nunique()} groups')
    _boxplot_pair_by_celltype(df, group_col, source, sort_by=sort_by)


# ------------- MISCELLANEOUS -------------
def _source_adata(source):
    """Look up the adata for `source_dropdown.value` -- 'all sources' is `adata` itself, every other option is `adata_{source}`."""
    return adata if source == 'all sources' else globals()[f'adata_{source}']

### Load h5ad from integration

In [ ]:
# ── Load integrated AnnData (used by all sections below) ──────────────────
print('Loading integrated.h5ad ... see `reference_building/README.md`  for description of `obs`')
adata = ad.read_h5ad(os.path.join(OUTPUT_DIR, h5ad_name))
print(f'  {adata.n_obs:,} cells · {adata.n_vars:,} genes')
print(f'  Sources: {dict(adata.obs["source"].value_counts())}')
print(f'  obs columns: {adata.obs.columns.tolist()}')
adata.obs['menon_DA'] = np.where(adata.obs['menon_author_cell_type'] == 'Dopaminergic', 'DA', 'non-DA')
adata.obs['siletti_DA'] = np.where(adata.obs['siletti_allen_da_vglut2'].astype(bool), 'DA', 'non-DA')
adata.obs['kamath_DA'] = np.where(adata.obs['cell_type_scanvi'].isin(_DA_PD_SUFFIXES), 'DA', 'non-DA')

# Gene symbol -> Ensembl ID mapping
_menon_cache_paths = glob.glob(f'{REFERENCE_DATA_DIR}/menon/intermediate/*_sn_only*.h5ad')
if _menon_cache_paths:
    _menon_var = sc.read_h5ad(_menon_cache_paths[0], backed='r').var
    adata.var['gene_symbol'] = _menon_var['feature_name'].reindex(adata.var_names)
    print(f"  Gene symbols resolved for {adata.var['gene_symbol'].notna().sum():,}/{adata.n_vars:,} "
          f"genes (via {_menon_cache_paths[0]})")
else:
    adata.var['gene_symbol'] = np.nan
    print("  Warning: no Menon SN-only cache found -- gene_symbol lookup unavailable.")

# Get each source AnnData
adata_wo_kamath = adata[adata.obs['source'] != "kamath"]
adata_kamath = adata[adata.obs['source'] == "kamath"]
adata_menon = adata[adata.obs['source'] == "menon"]
adata_siletti = adata[adata.obs['source'] == "siletti"]
adata_asap_cohort = adata[adata.obs['source'] == "asap"]

# Materialize Siletti's derived label columns once (NaN for non-Siletti cells).
if 'siletti_allen_da_vglut2' in adata.obs.columns:
    _siletti_mask = adata.obs['source'] == 'siletti'
    adata.obs['siletti_allen_da_vglut2_label'] = np.nan
    adata.obs.loc[_siletti_mask, 'siletti_allen_da_vglut2_label'] = (
        adata.obs.loc[_siletti_mask, 'siletti_allen_da_vglut2']
        .map({True: 'DA VGLUT2 (Allen)', False: 'non-DA'})
    )
    adata.obs['siletti_allen_cluster'] = np.nan
    adata.obs.loc[_siletti_mask, 'siletti_allen_cluster'] = (
        adata.obs.loc[_siletti_mask, 'siletti_allen_subcluster']
        .map(_load_siletti_subcluster_to_cluster())
    )
    # Coarse cell type -- abbreviated prefix of siletti_allen_cluster (e.g. 'Astro_60' -> 'Astro').
    adata.obs['siletti_allen_cell_type'] = (
        adata.obs['siletti_allen_cluster'].str.replace(r'_\d+$', '', regex=True)
    )
    print(
        f"  Derived siletti_allen_da_vglut2_label, siletti_allen_cluster, "
        f"siletti_allen_cell_type "
        f"({int(adata.obs['siletti_allen_cluster'].notna().sum()):,} Siletti cells)"
    )

## 1 — UMAP Explorer

Scatter plot of the scANVI-latent UMAP (`obsm['X_umap']`).

## Select a source or `all sources` to plot

In [ ]:
display(source_dropdown)

In [ ]:
_plot_umaps_categ(adata=_source_adata(source_dropdown.value))

### (Optional) -- Use lasso tool to select cells/dots from UMAP and save their barcodes to `selected_barcodes`

In [ ]:
selected_barcodes = _plot_umaps_categ(adata=_source_adata(source_dropdown.value))

In [ ]:
### Save list of barcodes selected with lasso tool
for colourby in selected_barcodes.keys():
    if len(selected_barcodes[colourby]) > 0:
        out_path = Path(OUTPUT_DIR) / f'selected_barcodes_{colourby}.txt'
        with open(out_path, 'w') as f:
            for barcode in selected_barcodes[colourby]:
                f.write(barcode + '\n')

        print(f"Wrote {len(selected_barcodes[colourby]):,} barcodes -> {out_path}")

### Gene Expression UMAP

In [ ]:
genes_to_plot = ['TH', 'CALB1', 'SOX6']
_plot_umaps_by_gene = functools.partial(plot_umaps_by_gene)
_plot_umaps_by_gene(adata=_source_adata(source_dropdown.value), panel_genes=genes_to_plot)

## 2 — Label Agreement

In [ ]:
# Explicit Output()+observe() rebuild -- label_agreement_widget's row/col
# dropdown *options* depend on source, so nesting source_dropdown inside its
# own @widgets.interact wouldn't rebuild those; only a full rebuild does.
_label_agreement_out = widgets.Output()


def _redraw_label_agreement(_=None):
    _label_agreement_out.clear_output(wait=True)
    with _label_agreement_out:
        label_agreement_widget(adata, source=source_dropdown.value)


source_dropdown.observe(_redraw_label_agreement, names='value')
display(source_dropdown, _label_agreement_out)
_redraw_label_agreement()

## 3 — Read Depth QC — UMI counts & genes measured per cell type

Per-cell UMI counts (total raw counts) and genes-measured (nonzero genes). Note that Kamath and Siletti DA's show higher number of UMIs than other cell types.

In [ ]:
@widgets.interact(source=qc_source_dropdown, sort_by=qc_sort_by_dropdown)
def _run_qc_boxplot(source, sort_by):
    qc_boxplot_widget(source, sort_by=sort_by)